## Without web search

In [1]:
from pathlib import Path
import sys

from dotenv import load_dotenv

def find_project_root():
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / "minimax_model.py").exists():
            return path
    raise FileNotFoundError("Could not find minimax_model.py")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env", override=True)

True

In [2]:
from langchain.agents import create_agent
from minimax_model import get_model

agent = create_agent(
    model=get_model()
)

d:\langchain\lca-lc-foundations\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [6]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Who is the current mayor of San Francisco?")

response = agent.invoke(
    {"messages": [question]}
)

In [7]:
print(response['messages'][-1].content)

<think>
The user asks: "Who is the current mayor of San Francisco?" This is a factual question about current office holder. I need to answer with the name. As of current date (2026-01-25). Let's consider the latest known info. As of 2024, London Breed was mayor. In 2025 there were elections. Actually we need to be careful: The last known information is up to 2024. But we can try to answer with the best knowledge. Let's check if any changes: London Breed was re-elected? I recall London Breed won a second term in November 2023. But maybe there was a recall or new election in 2025. Actually I recall that in November 2023, London Breed was re-elected. She took office for a second term in January 2024. There was a potential recall effort, but not sure if successful. As of 2025, it's still likely London Breed. However, there might be upcoming election in 2025 for mayor? No, mayor elections in SF are in odd-numbered years? Actually SF mayoral elections are held in odd-numbered years (e.g., 20

## Add web search tool

In [5]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of San Francisco?")

{'query': 'Who is the current mayor of San Francisco?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Mayor_of_San_Francisco',
   'title': 'Mayor of San Francisco - Wikipedia',
   'content': 'The current mayor is Democrat Daniel Lurie.',
   'score': 0.95324475,
   'raw_content': None},
  {'url': 'https://ballotpedia.org/Daniel_Lurie',
   'title': 'Daniel Lurie - Ballotpedia',
   'content': 'Daniel Lurie is the Mayor of San Francisco in California. He assumed office on January 8, 2025. His current term ends on January 8, 2029.',
   'score': 0.88799405,
   'raw_content': None},
  {'url': 'https://apnews.com/article/san-francisco-new-mayor-liberal-city-81ea0a7b37af6cbb68aea7ef5cc6a4f0',
   'title': "San Francisco's new mayor is starting to unite the fractured city",
   'content': 'San Francisco Mayor Daniel Lurie, a political newcomer and Levi Strauss heir, has marked his first 100 days with a hands-on, business-friendly 

In [8]:
from minimax_model import get_model
agent = create_agent(
    model=get_model(),
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of San Francisco?")

response = agent.invoke(
    {"messages": [question]}
)

In [9]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='Who is the current mayor of San Francisco?', additional_kwargs={}, response_metadata={}, id='803afcfa-a036-4642-a5c5-e3d9b124930f'),
 AIMessage(content='<think>\nWe need to answer who is the current mayor of San Francisco. As of my knowledge cutoff in 2021, London Breed was the mayor. But we are in 2025 now. We need to confirm the current mayor (as of 2025). The user may be asking for the present date (2026-01-25). The knowledge cutoff is 2021, but we can browse the web for up-to-date info. The system does not restrict browsing. We can use the web_search tool.\n\nThus we should do a web search for "current mayor of San Francisco 2025". Let\'s do that.\n</think>\n\n\n', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 160, 'prompt_tokens': 197, 'total_tokens': 357, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 130, 'rejected_prediction_tokens': None}, 'prompt_

trace: https://smith.langchain.com/public/59432173-0dd6-49e8-9964-b16be6048426/r